# What must be known

The Rietveld fit takes the pulse pattern, the open beam and the edge shape
(the nbragg response) as known. This notebook estimates how errors in these
inputs bias the result, and what freeing them costs in precision.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nbragg
import frame_overlap as fo

# the material, as in nbragg: 2 cm of alpha-iron
xs = nbragg.CrossSection(materials={"iron": {"mat": "Fe_sg229_Iron-alpha.ncmat"}})
model = nbragg.TransmissionModel(xs, response="jorgensen_inv", tof_length=9.0, vary_response=False)
model.params["thickness"].set(value=2.0)     # cm
model.params["α0"].set(value=3000.0)         # moderator response (rise), close to a water moderator
model.params["β0"].set(value=50.0)           # moderator response (decay)
sample = fo.Sample(model, free=["thickness", "strain"])

# the instrument: 9 m flight path, 50 ms period, 10 us bins, a thermal moderator
grid = fo.TimeGrid(flight_path=9.0, frequency=20.0, bin_width=10e-6)
beam = fo.OpenBeam.maxwellian(grid, flux=5e6, area=0.03)
hours = 8

In [2]:
pattern = fo.Pattern.times([0.0, 11.97e-3, 22.33e-3, 35.48e-3], width=200e-6)
exp = fo.Experiment(sample, beam, pattern, time=hours * 3600)
cases = {
    "one pulse 1 us late": fo.Pattern(pattern.starts + np.r_[0, 1e-6, 0, 0], 200e-6),
    "all pulses 1 us late (time zero)": fo.Pattern(pattern.starts + 1e-6, 200e-6),
    "one pulse 1 % stronger, same total": fo.Pattern(pattern.starts, 200e-6, intensity=[1.01, 1 - 0.01 / 3, 1 - 0.01 / 3, 1 - 0.01 / 3]),
    "all pulses 1 % stronger (normalisation)": fo.Pattern(pattern.starts, 200e-6, intensity=1.01),
    "pulses 1 % wider, same centre and neutrons": fo.Pattern((pattern.starts - 1e-6) % grid.period, 202e-6, intensity=200 / 202),
    "pulses 1 % longer, same start": fo.Pattern(pattern.starts, 202e-6),
}
import pandas as pd
pd.DataFrame({k: fo.pattern_bias(exp, p) for k, p in cases.items()}).T.round(2)

,thickness,strain
one pulse 1 us late,-0.17,0.42
all pulses 1 us late (time zero),-1.02,1.64
"one pulse 1 % stronger, same total",0.01,0.06
all pulses 1 % stronger (normalisation),-20.94,-0.23
"pulses 1 % wider, same centre and neutrons",0.08,-0.13
"pulses 1 % longer, same start",-21.98,1.33


The table gives the bias in units of the standard deviation of each
parameter. Pulse times matter for the strain (a digital chopper sets them to
far better than a microsecond), and a common delay is a time-zero error of any
time-of-flight measurement, which cancels in strain measured against a
reference sample. A pulse that ends later moves its centre and acts like a
delay of half the change; a pulse that is wider about the same centre hardly
matters. The normalisation matters for the thickness.

## The cost of freeing what is assumed known

In [3]:
model_r = nbragg.TransmissionModel(xs, response="jorgensen_inv", tof_length=9.0, vary_response=True)
model_r.params["thickness"].set(value=2.0)
model_r.params["α0"].set(value=3000.0)
model_r.params["β0"].set(value=50.0)
rows = {}
for name, free in {"thickness, strain": ["thickness", "strain"],
                   "+ normalisation": ["thickness", "strain", "norm"],
                   "+ moderator response": ["thickness", "strain", "α0", "β0"]}.items():
    s = fo.Sample(model_r, free=free)
    rows[name] = fo.Experiment(s, beam, pattern, time=hours * 3600).precision().set_index("parameter").sigma
pd.DataFrame(rows).T

parameter,norm,strain,thickness,α0,β0
"thickness, strain",NaN,0.000087,0.000442,NaN,NaN
+ normalisation,0.00508,0.000088,0.004720,NaN,NaN
+ moderator response,NaN,0.000269,0.000445,6964.949944,1.382307


Freeing the normalisation costs most of the thickness precision, and
freeing the moderator response costs most of the strain precision: both should
be calibrated in separate measurements.

The background, a common time offset and the relative pulse intensities can
also be treated as unknown, in the bound and in ``RietveldFit``
(``fit_background``, ``fit_t0``, ``fit_intensities``):

In [4]:
pd.DataFrame({"known": exp.precision().set_index("parameter").sigma,
              "+ background": exp.precision(background=True).set_index("parameter").sigma,
              "+ time offset": exp.precision(t0=True).set_index("parameter").sigma,
              "+ pulse intensities": exp.precision(intensities=True).set_index("parameter").sigma}).T

parameter,background,intensity_2,intensity_3,intensity_4,strain,t0,thickness
known,NaN,NaN,NaN,NaN,0.000087,NaN,0.000442
+ background,0.000069,NaN,NaN,NaN,0.000087,NaN,0.000442
+ time offset,NaN,NaN,NaN,NaN,0.000114,5.100116e-07,0.000501
+ pulse intensities,NaN,0.001005,0.000999,0.001,0.000087,NaN,0.000716
